# 02 — Baseline model

**Goal:** establish the number every real model has to beat. Without this, a reported accuracy has no context — see the "Why this matters" callout on baselines in the root README.

**Input:** `data/raw/wine.csv`, split with the project's fixed seed (`config/default.yaml`).

**Conclusion:** _(written after running the cells below)_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ucu-ai-course/project-template/blob/main/notebooks/02-baseline-model.ipynb)

In [1]:
import sys

# Safe to run locally too: this block only does anything inside Google Colab, where
# the repo isn't already cloned and the package isn't already installed.
if "google.colab" in sys.modules:
    get_ipython().system("git clone https://github.com/ucu-ai-course/project-template.git")
    get_ipython().run_line_magic("cd", "project-template")
    get_ipython().system("pip install -q -r requirements.txt")
    get_ipython().system("pip install -q -e .")


In [2]:
from pathlib import Path
import os

# Notebooks are launched from notebooks/ (`jupyter lab notebooks/`), but every path in this
# project (config/, data/, models/, results/) is written relative to the repo root.
# Running this cell once makes every relative path below "just work", the same way it
# does for the CLI and scripts/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT


PosixPath('/Users/oleksandr/Documents/GitHub/project-template')

In [3]:
from wine_origin.config import load_config
from wine_origin.data import (
    load_raw_csv,
    split_features_target,
    train_test_split_stratified,
)
from wine_origin.evaluate import compute_metrics
from wine_origin.models import build_baseline
from wine_origin.utils import set_seed

config = load_config("config/default.yaml")
set_seed(config["seed"])

df = load_raw_csv(config["paths"]["raw_data"])
train_df, test_df = train_test_split_stratified(
    df, test_size=config["split"]["test_size"], seed=config["seed"]
)
X_train, y_train = split_features_target(train_df)
X_test, y_test = split_features_target(test_df)
len(X_train), len(X_test)

(142, 36)

In [4]:
baseline = build_baseline()
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

baseline_metrics = compute_metrics(y_test, y_pred)
{k: v for k, v in baseline_metrics.items() if k != "confusion_matrix"}

{'accuracy': 0.3888888888888889, 'macro_f1': 0.18666666666666668}

In [5]:
import pandas as pd

pd.DataFrame(
    baseline_metrics["confusion_matrix"],
    index=[f"true_{i}" for i in range(3)],
    columns=[f"pred_{i}" for i in range(3)],
)

,pred_0,pred_1,pred_2
true_0,0,12,0
true_1,0,14,0
true_2,0,10,0


## Conclusion

`DummyClassifier(strategy="most_frequent")` gets 38.9% accuracy and a macro-F1 of 0.19 — it always predicts the majority class (`class_1`), so every other class is a total miss (see the all-zero rows in the confusion matrix above). Any model in `03-model-comparison.ipynb` needs to clear this bar by a wide margin, on both metrics, to be worth using.